# Preprocess Voice and Extract Features

Read `inputs/voice.wav`, prepare mono 44.1 kHz audio, extract consonant texture, and analyze event timing, amplitude, and pitch. Filter background-like events and save the pitch shift used during synthesis.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from scipy.signal import butter, filtfilt

from voice_to_birdcall.paths import INPUTS_DIR, PROCESSED_DATA_DIR

SR = 44_100
CONSONANT_CUTOFF = 4_000
CONSONANT_ZCR_THRESHOLD = 0.1

voice_source = INPUTS_DIR / "voice.wav"
voice_path = PROCESSED_DATA_DIR / "voice.wav"
consonant_path = PROCESSED_DATA_DIR / "consonant.wav"
events_path = PROCESSED_DATA_DIR / "voice-events.json"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Prepare Source WAV

Read `inputs/voice.wav` with SoundFile, average channels to mono, and resample to 44.1 kHz only when needed. Write the prepared PCM16 audio to `data/processed/voice.wav`. No FFmpeg decoding step is needed; the source recording is preserved.

In [2]:
def prepare_voice_wav(source: Path, destination: Path) -> None:
    """Read WAV input and prepare mono 44.1 kHz PCM16 audio for analysis."""

    if not source.is_file():
        raise FileNotFoundError(f"Voice source not found: {source}")
    if source.suffix.lower() != ".wav":
        raise ValueError("Voice input must be a .wav file")
    if source.resolve() == destination.resolve():
        raise ValueError("The processed WAV must not overwrite the input recording")

    audio, sample_rate = sf.read(str(source), dtype="float32", always_2d=True)
    if len(audio) == 0:
        raise ValueError(f"Audio file is empty: {source.name}")
    if not np.all(np.isfinite(audio)):
        raise ValueError(f"Audio contains non-finite samples: {source.name}")
    audio = np.mean(audio, axis=1)
    if sample_rate != SR:
        audio = librosa.resample(audio, orig_sr=sample_rate, target_sr=SR)
    sf.write(str(destination), audio, SR, subtype="PCM_16")


prepare_voice_wav(voice_source, voice_path)
print(f"Prepared voice: {voice_path}")

Prepared voice: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\voice.wav


# Extract Consonant Texture

Keep high-energy frames with a zero-crossing rate above 0.1, isolate content above 4 kHz, and add short 15, 25, and 30 ms reflections.

In [3]:
def finite_audio(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if not np.all(np.isfinite(y)):
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y


def normalize_audio(y: np.ndarray) -> np.ndarray:
    y = finite_audio(y)
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    return y if peak < 1e-8 else y / peak


def highpass_filter(y: np.ndarray, sr: int, cutoff: float) -> np.ndarray:
    if len(y) < 32:
        return y
    b, a = butter(4, cutoff / (sr / 2), btype="highpass")
    padlen = min(3 * max(len(a), len(b)), len(y) - 1)
    return filtfilt(b, a, y, padlen=padlen)


def load_audio(path: Path) -> np.ndarray:
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio file is empty: {path.name}")
    return normalize_audio(y)


def save_wav(path: Path, y: np.ndarray) -> None:
    sf.write(str(path), finite_audio(y), SR, subtype="PCM_16")


def extract_consonants(y: np.ndarray) -> np.ndarray:
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=2048, hop_length=512)[0]
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    high_freq_y = highpass_filter(y, SR, cutoff=CONSONANT_CUTOFF)
    zcr_up = np.interp(np.linspace(0, 1, len(y)), np.linspace(0, 1, len(zcr)), zcr)
    rms_up = np.interp(np.linspace(0, 1, len(y)), np.linspace(0, 1, len(rms)), rms)
    threshold = np.percentile(rms_up, 65)
    mask = (zcr_up > CONSONANT_ZCR_THRESHOLD) & (rms_up > threshold)
    return high_freq_y * mask * 0.6


def apply_consonant_effects(part: np.ndarray) -> np.ndarray:
    if len(part) == 0 or np.max(np.abs(part)) < 1e-6:
        return part

    delayed = np.zeros_like(part)
    delay_30ms = int(0.03 * SR)
    if delay_30ms < len(part):
        delayed[delay_30ms:] = part[:-delay_30ms] * 0.4

    combined = part + delayed
    reverb = np.zeros_like(combined)
    feedback = 0.3
    for delay in [0.015, 0.025]:
        samples = int(delay * SR)
        if samples < len(combined):
            reverb[samples:] += combined[:-samples] * feedback
        feedback *= 0.7
    return (combined + reverb) * 0.5


voice = load_audio(voice_path)
consonant = apply_consonant_effects(extract_consonants(voice))
save_wav(consonant_path, consonant)

print(f"Voice duration: {len(voice) / SR:.2f} seconds")
print(f"Saved consonant texture to: {consonant_path}")

Voice duration: 3.72 seconds
Saved consonant texture to: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\consonant.wav


# Analyze and Select Voice Events

Keep the original onset timing, RMS amplitude, and relative pYIN pitch for accepted events. Validate each candidate using 30 ms of lookbehind and 150 ms of lookahead, retaining clear tonal sounds such as whistles. Separately measure recording F0 down to about 65 Hz using confidence >= 0.60 and at least 0.1 seconds of reliable frames. Save the actual pitch shift and selection diagnostics with the events. If all candidates are rejected, notebook 03 writes a silent mix.

In [4]:
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class VoiceAnalysisConfig:
    min_pitch_confidence: float = 0.60
    noise_guard: bool = True
    noise_margin_db: float = 6.0
    relative_energy_floor: float = 0.06
    lookbehind_seconds: float = 0.03
    lookahead_seconds: float = 0.15
    tonal_concentration: float = 0.70
    tonal_relative_floor: float = 0.20

    def __post_init__(self):
        if any(not np.isfinite(value) for value in asdict(self).values()):
            raise ValueError("Configuration values must be finite")
        if not 0 <= self.min_pitch_confidence <= 1:
            raise ValueError("min_pitch_confidence must be between 0 and 1")
        if self.lookahead_seconds <= 0 or self.lookbehind_seconds < 0:
            raise ValueError("Invalid event context window")
        if self.noise_margin_db < 0:
            raise ValueError("noise_margin_db must be nonnegative")
        for value in (
            self.relative_energy_floor,
            self.tonal_concentration,
            self.tonal_relative_floor,
        ):
            if not 0 <= value <= 1:
                raise ValueError("Energy/tonality fractions must be between 0 and 1")


def analyze_voice_controls(audio, candidate_events, *, sr=44_100, config=None):
    """Filter existing candidates and measure one confidence-gated recording F0.

    Backtracked onsets may precede the audible attack, so validation looks ahead.
    A strong narrowband tone can pass even when continuous (e.g. a whistle).
    This is a background heuristic, not an inference about human intention.
    """
    config = config or VoiceAnalysisConfig()
    y = np.asarray(audio, dtype=np.float32)
    if y.ndim != 1 or not len(y) or not np.all(np.isfinite(y)):
        raise ValueError("Expected a nonempty, finite mono waveform")
    if sr < 8_000:
        raise ValueError("Sample rate must be at least 8 kHz")
    # Reuse the same RMS frame/hop geometry as the original 44.1 kHz pipeline.
    hop = max(1, round(sr * 512 / 44_100))
    frame = max(256, round(sr * 2048 / 44_100))
    rms = librosa.feature.rms(y=y, frame_length=frame, hop_length=hop)[0]
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)
    noise = float(np.percentile(rms, 20))
    anchor = max(float(np.percentile(rms, 95)), float(np.max(rms)) * 0.25)
    threshold = max(
        1e-7, noise * 10 ** (config.noise_margin_db / 20), anchor * config.relative_energy_floor
    )

    spectrum = np.abs(librosa.stft(y, n_fft=frame, hop_length=hop)) ** 2
    frequencies = librosa.fft_frequencies(sr=sr, n_fft=frame)
    band = spectrum[(frequencies >= 80) & (frequencies <= min(10_000, sr / 2))]
    # Five bins around the strongest frequency avoid classifying tonal whistles
    # as broadband background simply because they are not human speech.
    peaks = np.argmax(band, axis=0)
    columns = np.arange(band.shape[1])
    tonal_power = np.zeros(band.shape[1], dtype=np.float64)
    for delta in range(-2, 3):
        index = peaks + delta
        valid = (index >= 0) & (index < band.shape[0])
        tonal_power[valid] += band[index[valid], columns[valid]]
    concentration = tonal_power / np.maximum(np.sum(band, axis=0), 1e-20)
    length = min(len(rms), len(concentration))
    rms, times, concentration = rms[:length], times[:length], concentration[:length]
    # A quiet but clear whistle need not match the loudest spoken syllable.
    # Require it to stand above the recording's floor, or (for a continuous
    # tonal recording without quiet frames) a fraction of the active level.
    tonal_floor = max(1e-7, min(anchor * config.tonal_relative_floor, noise * np.sqrt(2)))
    tonal = (concentration >= config.tonal_concentration) & (rms >= tonal_floor) & (rms > 1e-7)
    active = (rms >= threshold) | tonal

    accepted, rejected = [], []
    for event in candidate_events:
        time = float(event["time"])
        if not np.isfinite(time) or not 0 <= time <= len(y) / sr:
            raise ValueError("Candidate event time is outside the recording")
        start = max(0.0, time - config.lookbehind_seconds)
        stop = time + config.lookahead_seconds
        context = (times >= start) & (times <= stop)
        if not np.any(context):
            context[np.argmin(np.abs(times - time))] = True
        local_peak = float(np.max(rms[context]))
        keep = not config.noise_guard or bool(np.any(active[context]))
        if keep:
            accepted.append(dict(event))  # exact original times, amplitude and pitch
        else:
            rejected.append(
                {
                    "time": time,
                    "context_peak_rms": local_peak,
                    "reason": "weak/background-like event context",
                }
            )

    overall_pitch = None
    reliable_frames = 0
    if np.any(active):
        # Broader low end than original C3, so lower voices are not lost.
        # Downsampling keeps this extra recording-level measurement lightweight.
        pitch_sr = min(sr, 22_050)
        pitch_audio = y if pitch_sr == sr else librosa.resample(y, orig_sr=sr, target_sr=pitch_sr)
        pitch_hop = max(1, round(pitch_sr * 512 / 44_100))
        f0, voiced, confidence = librosa.pyin(
            pitch_audio,
            fmin=65.406,
            fmax=min(2_000.0, pitch_sr * 0.45),
            sr=pitch_sr,
            frame_length=2048,
            hop_length=pitch_hop,
        )
        pitch_times = librosa.frames_to_time(np.arange(len(f0)), sr=pitch_sr, hop_length=pitch_hop)
        active_at_pitch = np.interp(pitch_times, times, active.astype(float)) >= 0.5
        valid = (
            np.isfinite(f0) & voiced & (confidence >= config.min_pitch_confidence) & active_at_pitch
        )
        reliable_frames = int(np.count_nonzero(valid))
        if reliable_frames * pitch_hop / pitch_sr >= 0.10:
            overall_pitch = float(np.median(f0[valid]))
    return {
        "events": accepted,
        "recording_pitch_hz": overall_pitch,
        "shift_semitones": lower_register_offset(overall_pitch),
        "pitch_confident_frames": reliable_frames,
        "candidate_count": len(candidate_events),
        "accepted_count": len(accepted),
        "rejected_count": len(rejected),
        "rejected_events": rejected,
        "noise_floor_rms": noise,
        "energy_threshold_rms": threshold,
        "config": asdict(config),
    }


def lower_register_offset(pitch_hz):
    """High/unknown F0: -1 semitone; lower F0: smoothly down to -12 total.

    The 220 Hz reference is a tuning reference, never a gender classifier.
    Recordings at or below about 116.54 Hz reach the -12-semitone limit.
    """
    if pitch_hz is None or not np.isfinite(pitch_hz) or pitch_hz <= 0:
        return -1.0
    return float(np.clip(-1 + 12 * np.log2(min(pitch_hz, 220.0) / 220.0), -12, -1))


In [5]:
def get_voice_features(y: np.ndarray) -> tuple[list[dict], float]:
    onset_frames = librosa.onset.onset_detect(
        y=y,
        sr=SR,
        units="frames",
        backtrack=True,
        delta=0.08,
        wait=2,
    )
    onset_times = librosa.frames_to_time(onset_frames, sr=SR)
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=512)[0]
    rms_times = librosa.frames_to_time(np.arange(len(rms)), sr=SR, hop_length=512)
    f0, _, _ = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C3"),
        fmax=librosa.note_to_hz("C7"),
        sr=SR,
    )
    pitch_times = librosa.frames_to_time(np.arange(len(f0)), sr=SR)
    valid = ~np.isnan(f0)
    valid_pitch = f0[valid]
    median_pitch = float(np.median(valid_pitch)) if len(valid_pitch) else 440.0

    events = []
    for time in onset_times:
        amplitude = float(np.interp(time, rms_times, rms))
        pitch = float(
            np.interp(
                time,
                pitch_times[valid] if np.any(valid) else [0],
                f0[valid] if np.any(valid) else [median_pitch],
            )
        )
        events.append({"time": float(time), "amp": amplitude, "pitch": pitch})
    return events, median_pitch


events, median_pitch = get_voice_features(voice)
if not events:
    duration = len(voice) / SR
    events = [
        {"time": float(time), "amp": 0.7, "pitch": median_pitch}
        for time in np.arange(0.15, max(0.2, duration), 0.35)
    ]

controls = analyze_voice_controls(voice, events, sr=SR)
events = controls["events"]
analysis = {"sample_rate": SR, "median_pitch_hz": median_pitch, **controls}
events_path.write_text(json.dumps(analysis, indent=2), encoding="utf-8")

print(f"Median pitch: {median_pitch:.1f} Hz")
print(f"Accepted voice events: {len(events)} / {controls['candidate_count']}")
print(f"Recording pitch: {controls['recording_pitch_hz']} Hz")
print(f"Pitch shift: {controls['shift_semitones']:.2f} semitones")
print(f"Saved voice features to: {events_path}")
events[:10]

Median pitch: 359.5 Hz
Accepted voice events: 12 / 16
Recording pitch: 363.6359643480317 Hz
Pitch shift: -1.00 semitones
Saved voice features to: C:\Users\morph\GitHub\voice-to-birdcall\data\processed\voice-events.json


[{'time': 0.3250793650793651,
  'amp': 0.010920057073235512,
  'pitch': 365.74467430282664},
 {'time': 0.7662585034013606,
  'amp': 0.026105990633368492,
  'pitch': 365.74467430282664},
 {'time': 0.8591383219954648,
  'amp': 0.022848600521683693,
  'pitch': 365.74467430282664},
 {'time': 0.9171882086167801,
  'amp': 0.2928451895713806,
  'pitch': 361.5437384177477},
 {'time': 1.195827664399093,
  'amp': 0.025254415348172188,
  'pitch': 384.8813818384261},
 {'time': 1.4280272108843537,
  'amp': 0.14235322177410126,
  'pitch': 278.7883336231536},
 {'time': 1.6602267573696146,
  'amp': 0.08274157345294952,
  'pitch': 249.81097439559304},
 {'time': 2.147845804988662,
  'amp': 0.006337990052998066,
  'pitch': 382.8619182306804},
 {'time': 2.3336054421768706,
  'amp': 0.00801890529692173,
  'pitch': 391.99543598174927},
 {'time': 3.0998639455782313,
  'amp': 0.004118951037526131,
  'pitch': 213.0389447286838}]